### Setting up drive and dependencies



In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import zipfile
import pathlib
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

### Setting up Derm7pt

In [ ]:
DERM7PT_ZIP = '/content/drive/MyDrive/derm7pt/release_v0.zip'
LOCAL_DERM7PT = '/content/derm7pt'
os.makedirs(LOCAL_DERM7PT, exist_ok=True)

if not os.path.exists(LOCAL_DERM7PT) or not os.listdir(LOCAL_DERM7PT):
    print('Unzipping Derm7pt...')
    with zipfile.ZipFile(DERM7PT_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DERM7PT)
else:
    print('Derm7pt already unzipped, skipping.')

matches = list(pathlib.Path(LOCAL_DERM7PT).rglob('meta.csv'))
if not matches:
    raise FileNotFoundError('Could not find meta.csv after unzipping Derm7pt.')
LOCAL_DERM7PT = str(matches[0].parent.parent)
print(f'Derm7pt root: {LOCAL_DERM7PT}')

derm7pt_meta = pd.read_csv(os.path.join(LOCAL_DERM7PT, 'meta', 'meta.csv'))
derm7pt_img_dir = Path(LOCAL_DERM7PT) / 'images'

# working out what all the diagnosis values are
print("\nUnique Derm7pt 'diagnosis' values:")
print(derm7pt_meta['diagnosis'].value_counts())

### Setting up HAM10000

In [ ]:
drive_dir = Path('/content/drive/MyDrive/ham10000_dataset')
ham_data_dir = Path('dataset/ham10000')
os.makedirs(ham_data_dir, exist_ok=True)

if drive_dir.exists():
    drive_jpg_count = len(list(drive_dir.glob('*.jpg')))
    print(f"Images found on Drive: {drive_jpg_count}")
    if drive_jpg_count == 0:
        raise FileNotFoundError(f"No .jpg files found in {drive_dir}.")

    existing_count = len(list(ham_data_dir.glob('*.jpg')))
    if existing_count < drive_jpg_count:
        print("Copying HAM10000 images (resumable)...")
        src_files = list(drive_dir.glob('*.jpg'))
        for f in tqdm(src_files):
            dst = ham_data_dir / f.name
            if not dst.exists():
                shutil.copy(f, dst)
        !cp /content/drive/MyDrive/ham10000_dataset/HAM10000_metadata.csv dataset/ham10000/ 2>/dev/null || true
    jpg_files = list(ham_data_dir.glob('*.jpg'))
else:
    raise FileNotFoundError(f"{drive_dir} not found — set up HAM10000 on Drive first.")

print(f"Total HAM10000 .jpg files found: {len(jpg_files)}")

ham_meta = pd.read_csv(ham_data_dir / 'HAM10000_metadata.csv')
valid_ids = set(f.stem for f in jpg_files)
ham_meta = ham_meta[ham_meta['image_id'].isin(valid_ids)].reset_index(drop=True)
print("HAM10000 class distribution:")
print(ham_meta['dx'].value_counts())

### Standardising each dataset's diagnosis labels

In [ ]:
HAM_CLASS_MAP = {'mel': 'melanoma', 'nv': 'nevus'}

DERM7PT_DIAGNOSIS_MAP = {  # taken from printing the diagnoses in the Derm7pt data, then choosing the ones that were melanoma or nevus
    'melanoma': 'melanoma',
    'melanoma (in situ)': 'melanoma',
    'melanoma (less than 0.76 mm)': 'melanoma',
    'melanoma (0.76 to 1.5 mm)': 'melanoma',
    'melanoma (more than 1.5 mm)': 'melanoma',
    'melanoma metastasis': 'melanoma',
    'clark nevus': 'nevus',
    'reed or spitz nevus': 'nevus',
    'blue nevus': 'nevus',
    'combined nevus': 'nevus',
    'congenital nevus': 'nevus',
    'dermal nevus': 'nevus',
    'recurrent nevus': 'nevus',
}

derm7pt_meta['common_class'] = derm7pt_meta['diagnosis'].map(DERM7PT_DIAGNOSIS_MAP)
unmapped = derm7pt_meta[derm7pt_meta['common_class'].isna()]['diagnosis'].unique()
if len(unmapped) > 0:
    print(f"WARNING: {len(unmapped)} unmapped Derm7pt diagnosis values — these rows will be excluded:")
    print(unmapped)

ham_meta['common_class'] = ham_meta['dx'].map(HAM_CLASS_MAP)

### Getting a generic pretrained model to look at the images

In [ ]:
embed_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
embed_model.fc = nn.Identity()
embed_model = embed_model.to(device).eval()

embed_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def get_embedding(img_path, crop_border=False, crop_amount=25):
    img = Image.open(img_path).convert('RGB')
    if crop_border:
        w, h = img.size
        img = img.crop((crop_amount, crop_amount, w - crop_amount, h - crop_amount))
    x = embed_transform(img).unsqueeze(0).to(device)
    return embed_model(x).cpu().numpy().flatten()

### Per-class similarity matching

In [ ]:
# HAM10000 counts
ham_melanoma_count = ham_meta[ham_meta['common_class'] == 'melanoma'].shape[0]
ham_nevus_count = ham_meta[ham_meta['common_class'] == 'nevus'].shape[0]

# Derm7pt counts
derm7pt_melanoma_count = derm7pt_meta[derm7pt_meta['common_class'] == 'melanoma'].shape[0]
derm7pt_nevus_count = derm7pt_meta[derm7pt_meta['common_class'] == 'nevus'].shape[0]

def build_ham_embedding_bank(ham_meta, target_class, img_dir, pool_size):
    candidates = ham_meta[ham_meta['common_class'] == target_class]['image_id'].tolist()
    if len(candidates) > pool_size:
        candidates = list(np.random.choice(candidates, pool_size, replace=False))
    bank = {}
    print(f"Embedding {len(candidates)} HAM10000 '{target_class}' images...")
    for image_id in tqdm(candidates):
        path = img_dir / f"{image_id}.jpg"
        if path.exists():
            bank[image_id] = get_embedding(path, crop_border=False)
    return bank

def match_derm7pt_to_ham(derm7pt_meta, target_class, ham_bank, derm7pt_img_dir, n_samples, top_n=100):
    candidates = derm7pt_meta[derm7pt_meta['common_class'] == target_class]
    if len(candidates) > n_samples:
        candidates = candidates.sample(n_samples, random_state=0)

    matches = {}
    print(f"Matching {len(candidates)} Derm7pt '{target_class}' images against HAM10000 bank...")
    for _, row in tqdm(candidates.iterrows(), total=len(candidates)):
        derm_path = derm7pt_img_dir / row['derm']
        if not derm_path.exists():
            continue
        query_emb = get_embedding(derm_path, crop_border=True)

        best_id, best_sim = None, -1
        for ham_id, ham_emb in ham_bank.items():
            sim = np.dot(query_emb, ham_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(ham_emb))
            if sim > best_sim:
                best_id, best_sim = ham_id, sim

        matches[row['derm']] = (best_id, best_sim)
    top_matches = dict(sorted(matches.items(), key=lambda item: -item[1][1])[:top_n])
    return top_matches
# --- Run for melanoma ---
ham_bank_mel = build_ham_embedding_bank(ham_meta, 'melanoma', ham_data_dir, ham_melanoma_count)
matches_melanoma = match_derm7pt_to_ham(derm7pt_meta, 'melanoma', ham_bank_mel, derm7pt_img_dir, derm7pt_melanoma_count, top_n=100)

# --- Run for nevus ---
ham_bank_nv = build_ham_embedding_bank(ham_meta, 'nevus', ham_data_dir, ham_nevus_count)
matches_nevus = match_derm7pt_to_ham(derm7pt_meta, 'nevus', ham_bank_nv, derm7pt_img_dir, derm7pt_nevus_count, top_n=100)

print("\n--- Melanoma matches ---")
print(matches_melanoma)
print("\n--- Nevus matches ---")
print(matches_nevus)

### Saving matched pairs to a CSV

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/derm7pt/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def matches_to_df(matches, target_class):
    return pd.DataFrame(
        [
            {
                'derm7pt_image': derm_file,
                'ham10000_image_id': ham_id,
                'cosine_similarity': sim,
                'class': target_class,
            }
            for derm_file, (ham_id, sim) in matches.items()
        ]
    )

df_melanoma = matches_to_df(matches_melanoma, 'melanoma')
df_nevus = matches_to_df(matches_nevus, 'nevus')

df_melanoma = df_melanoma.sort_values(by='cosine_similarity', ascending=False).head(100)
df_nevus = df_nevus.sort_values(by='cosine_similarity', ascending=False).head(100)

df_melanoma.to_csv(os.path.join(OUTPUT_DIR, 'matched_pairs_melanoma_TOP100_new.csv'), index=False)
df_nevus.to_csv(os.path.join(OUTPUT_DIR, 'matched_pairs_nevus_TOP100_new.csv'), index=False)

print('Saved top 100 matched pairs (sorted by cosine similarity) to Drive.')